In [1]:
!pip install -r requirements.txt

In [2]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Quick check
import os
print("OpenAI key set:", bool(os.environ.get("OPENAI_API_KEY")))
print("LangSmith key set:", bool(os.environ.get("LANGCHAIN_API_KEY")))

OpenAI key set: True
LangSmith key set: True


Import Statements


In [3]:
import os
import re
import json
from typing import List, Dict, Any, Optional, Tuple
from enum import Enum
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# LangChain imports for tracing
from langsmith import Client
from langchain_core.tracers.context import tracing_v2_enabled

# Initialize LangSmith client
client = Client()



##Set up LLM and Stopwords

In [4]:
# OpenRouter model identifier
LLM_MODEL = "openai/gpt-5-nano"

# Use OpenAI embeddings via OpenRouter
EMBEDDING_MODEL = "text-embedding-3-small"

SIMILARITY_THRESHOLD = 0.6
N_RETRIEVAL_RESULTS = 1

stop_words = set(stopwords.words('english'))
print(f"Loaded {len(stop_words)} stopwords")

Loaded 198 stopwords


In [5]:
def get_llm():
    """
    Initialize the LLM using OpenRouter as the provider.
    OpenRouter acts as a proxy to various models including GPT-5-Nano.
    """
    llm = ChatOpenAI(
        model=LLM_MODEL,
        openai_api_base="https://openrouter.ai/api/v1",
        openai_api_key=os.environ["OPENAI_API_KEY"],
        temperature=0,
        max_tokens=500,
        default_headers={
            "HTTP-Referer": "https://medicare-chatbot.local",
            "X-Title": "MediCare Chatbot"
        }
    )
    return llm

def get_embeddings():
    """Initialize embeddings via OpenRouter"""
    embeddings = OpenAIEmbeddings(
        model=EMBEDDING_MODEL,
        openai_api_base="https://openrouter.ai/api/v1",
        openai_api_key=os.environ["OPENAI_API_KEY"]
    )
    return embeddings

llm = get_llm()
embeddings = get_embeddings()

# Quick test to make sure it works
test_response = llm.invoke("Say 'Hello' in a single word")
print(f"LLM test response: {test_response.content}")

LLM test response: Hello


##Patient Database

In [6]:
PATIENTS_DB = { 
    "P001": { 
        "name": "María García", 
        "email": "maria.garcia@email.com", 
        "phone": "+34 612 345 678", 
        "date_of_birth": "1985-03-15", 
        "membership_type": "Premium", 
        "balance_due": 45.00, 
        "upcoming_appointments": [ 
            { 
                "date": "2024-02-15", 
                "time": "10:30", 
                "doctor": "Dr. Fernández", 
                "specialty": "General Medicine", 
                "status": "confirmed" 
            }, 
            { 
                "date": "2024-03-01", 
                "time": "16:00", 
                "doctor": "Dr. López", 
                "specialty": "Dermatology", 
                "status": "pending_confirmation" 
            } 
        ], 
        "last_visit": "2024-01-20", 
        "primary_doctor": "Dr. Fernández", 
        "allergies": ["Penicillin"], 
        "insurance_provider": "Sanitas" 
    }, 
    "P002": { 
        "name": "Carlos Rodríguez", 
        "email": "carlos.r@email.com", 
        "phone": "+34 698 765 432", 
        "date_of_birth": "1972-11-28", 
        "membership_type": "Basic", 
        "balance_due": 0.00, 
        "upcoming_appointments": [], 
        "last_visit": "2023-11-05", 
        "primary_doctor": "Dr. Martín", 
        "allergies": [], 
        "insurance_provider": "Adeslas" 
    }, 
    "P003": { 
        "name": "Ana Belén Torres", 
        "email": "anabelen.t@email.com", 
        "phone": "+34 655 123 789", 
        "date_of_birth": "1990-07-22", 
        "membership_type": "Premium", 
        "balance_due": 120.50, 
        "upcoming_appointments": [ 
            { 
                "date": "2024-02-10", 
                "time": "09:00", 
                "doctor": "Dr. Sánchez", 
                "specialty": "Gynecology", 
                "status": "confirmed" 
            } 
        ], 
        "last_visit": "2024-01-28", 
        "primary_doctor": "Dr. Sánchez", 
        "allergies": ["Ibuprofen", "Latex"], 
        "insurance_provider": "Mapfre" 
    } 
} 

Clinic FAQs

In [7]:
CLINIC_FAQS = { 
    "FAQ001": { 
        "category": "Appointments", 
        "question": "How can I schedule an appointment?", 
        "answer": "You can schedule an appointment through our online portal at medicare-plus.com/appointments, by calling our reception at +34 911 234 567 (Monday to Friday, 8:00-20:00), or through this chat by requesting to speak with our scheduling team." 
    }, 
    "FAQ002": { 
        "category": "Appointments", 
        "question": "What is the cancellation policy?", 
        "answer": "Appointments must be cancelled at least 24 hours in advance to avoid a €20 cancellation fee. Premium members can cancel up to 12 hours before without penalty. To cancel, use our online portal or call reception." 
    }, 
    "FAQ003": { 
        "category": "Appointments", 
        "question": "How early should I arrive for my appointment?", 
        "answer": "Please arrive 15 minutes before your scheduled appointment time. First-time patients should arrive 30 minutes early to complete registration paperwork." 
    }, 
    "FAQ004": { 
        "category": "Payments", 
        "question": "What payment methods do you accept?", 
        "answer": "We accept cash, credit/debit cards (Visa, Mastercard, American Express), bank transfers, and payment through major insurance providers (Sanitas, Adeslas, Mapfre, DKV, Asisa). Payment plans are available for treatments exceeding €500." 
    }, 
    "FAQ005": { 
        "category": "Payments", 
        "question": "How can I check my balance or pay outstanding bills?", 
        "answer": "You can view and pay your balance through our patient portal at medicare-plus.com/billing, by phone at +34 911 234 568 (billing department), or in person at our reception desk. We send monthly statements via email for any outstanding balances." 
    }, 
    "FAQ006": { 
        "category": "Insurance", 
        "question": "Which insurance providers do you work with?", 
        "answer": "We work with Sanitas, Adeslas, Mapfre, DKV, and Asisa. Coverage varies by plan. Please bring your insurance card to every visit. We recommend calling your insurance provider to verify coverage before specialized procedures." 
    }, 
    "FAQ007": { 
        "category": "Services", 
        "question": "What medical specialties are available at the clinic?", 
        "answer": "MediCare Plus offers: General Medicine, Pediatrics, Gynecology, Dermatology, Cardiology, Traumatology, Psychology, and Nutrition. Some specialists are only available on specific days. Contact reception for specialist schedules." 
    }, 
    "FAQ008": { 
        "category": "Services", 
        "question": "Do you offer telemedicine consultations?", 
        "answer": "Yes, we offer video consultations for General Medicine, Psychology, Nutrition, and follow-up appointments. Telemedicine is available for Premium members at no extra cost. Basic members pay €15 per video consultation. Book through our portal or call reception." 
    }, 
    "FAQ009": { 
        "category": "Hours & Location", 
        "question": "What are the clinic's operating hours?", 
        "answer": "Regular hours: Monday to Friday 8:00-20:00, Saturday 9:00-14:00. Closed on Sundays and public holidays. Extended hours available by appointment for Premium members." 
    }, 
    "FAQ010": { 
        "category": "Hours & Location", 
        "question": "Where is the clinic located?", 
        "answer": "We are located at Calle Serrano 125, 28006 Madrid. Nearest metro:Núñez de Balboa (Line 5, 9). Paid parking available at Parking Serrano (2-minute walk). Wheelchair accessible entrance on Calle Claudio Coello." 
    }, 
    "FAQ011": { 
        "category": "Emergencies", 
        "question": "Do you handle medical emergencies?", 
        "answer": "MediCare Plus is NOT an emergency facility. For medical emergencies, call 112 or go to the nearest hospital emergency room. For urgent but non-emergency same-day care, Premium members can request urgent appointments subject to availability." 
    }, 
    "FAQ012": { 
        "category": "Membership", 
        "question": "What is the difference between Basic and Premium membership?", 
        "answer": "Basic membership: Standard appointment scheduling, access to all specialists, email support. Premium membership (€29/month): Priority scheduling, 12hour cancellation policy, free telemedicine, extended hours access, dedicated phone support line, and 10% discount on non-covered services." 
    }, 
    "FAQ013": { 
        "category": "Medical Records", 
        "question": "How can I access my medical records?", 
        "answer": "Access your records through our patient portal at medicareplus.com/records. You can also request printed copies at reception (€5 processing fee, ready within 48 hours). For records to be sent to another provider, submit a signed authorization form." 
    }, 
    "FAQ014": { 
        "category": "Prescriptions", 
        "question": "How do I request prescription refills?", 
        "answer": "Prescription refills can be requested through the patient portal or by calling your doctor's office directly. Please allow 48-72 hours for processing. Controlled substances require an in-person appointment." 
    }, 
    "FAQ015": { 
        "category": "COVID-19", 
        "question": "What COVID-19 protocols are in place?", 
"answer": "Masks are optional but recommended in waiting areas. Hand sanitizer stations are available throughout the clinic. If you have COVID-19 symptoms, please call before your visit to arrange appropriate care. Telemedicine is recommended for patients with respiratory symptoms." 
} 
}

##Intent Detection (Layer 1)

In [8]:
class Intent(Enum):
    EMERGENCY = "emergency"
    PERSONAL_DATA = "personal_data"
    FAQ = "faq"
    UNCLEAR = "unclear"

In [9]:
def enhanced_detect_intent_with_llm(message: str, conversation_history: List[Dict[str, str]]) -> Intent:
    """
    Enhanced intent detection with better prompt
    """
    history_context = ""
    if conversation_history:
        history_context = "Previous conversation:\n"
        for turn in conversation_history[-3:]:
            role = "User" if turn["role"] == "user" else "Assistant"
            history_context += f"{role}: {turn['content']}\n"

    prompt = f"""
    You are an intent classifier for MediCare Plus medical clinic.
    
    {history_context}
    
    Classify the user's current message into exactly one of these categories:
    
    1. EMERGENCY: Medical emergencies, urgent symptoms, life-threatening situations that require immediate 911/112.
    2. PERSONAL_DATA: Questions about the user's personal information (my balance, my appointment, my doctor, my insurance, etc.)
    3. FAQ: General questions about clinic operations, services, policies, hours, location, etc.
    4. UNCLEAR: Anything else, including questions about specific medical procedures we don't offer
    
    Current message: "{message}"
    
    Think step by step:
    1. Is this a medical emergency? If yes, EMERGENCY
    2. Is this about the user's personal medical data/account? If yes, PERSONAL_DATA  
    3. Is this a general question about clinic operations? If yes, FAQ
    4. Otherwise, UNCLEAR
    
    Respond with ONLY the category name: EMERGENCY, PERSONAL_DATA, FAQ, or UNCLEAR
    """

    try:
        response = llm.invoke(prompt)
        intent_str = response.content.strip().upper()

        # Map to Intent enum
        if "EMERGENCY" in intent_str:
            return Intent.EMERGENCY
        elif "PERSONAL_DATA" in intent_str:
            return Intent.PERSONAL_DATA
        elif "FAQ" in intent_str:
            return Intent.FAQ
        else:
            return Intent.UNCLEAR

    except Exception as e:
        print(f"LLM intent detection failed: {e}")
        return detect_intent_rule_based(message, conversation_history)

# Also need the fallback function if missing
def detect_intent_rule_based(message: str, conversation_history: List[Dict[str, str]]) -> Intent:
    """
    Fallback rule-based intent detection
    """
    message_lower = message.lower()

    # Emergency detection
    emergency_keywords = [
        "emergency", "urgent", "severe chest pain", "heart attack", "stroke",
        "bleeding heavily", "unconscious", "can't breathe", "not breathing",
        "dying", "ambulance", "severe pain", "chest pain", "having severe",
        "call 112", "help me", "critical", "life threatening", "911"
    ]

    if any(keyword in message_lower for keyword in emergency_keywords):
        return Intent.EMERGENCY

    # Personal data detection
    personal_indicators = [
        "my ", "do i owe", "my balance", "my appointment", "who is my",
        "what is my", "when is my", "my doctor", "my insurance", "my records",
        "my prescription", "i owe", "my upcoming", "my primary", "my next visit",
        "my allergies", "my last visit", "my membership", "my bill"
    ]

    if any(indicator in message_lower for indicator in personal_indicators):
        return Intent.PERSONAL_DATA

    # FAQ detection - check if message relates to any FAQ
    faq_keywords = set()
    for faq_id, faq in CLINIC_FAQS.items():
        question_words = set(faq["question"].lower().split())
        faq_keywords.update(question_words)

    message_words = set(message_lower.split())
    stop_words = set(stopwords.words('english'))
    relevant_words = message_words.intersection(faq_keywords) - stop_words

    if len(relevant_words) >= 2:
        return Intent.FAQ

    return Intent.UNCLEAR

# Set detect_intent to use the LLM version
detect_intent = enhanced_detect_intent_with_llm

print("✓ detect_intent_with_llm redefined and detect_intent set")

✓ detect_intent_with_llm redefined and detect_intent set


## RAG Logic (Layer 2)

In [10]:
def retrieve_faq_with_llm(message: str) -> Optional[Dict[str, Any]]:
    """
    Use LLM to find the best FAQ match
    """
    # Create a list of FAQs for the LLM
    faq_list = []
    for faq_id, faq in CLINIC_FAQS.items():
        faq_list.append(f"[{faq_id}] {faq['question']} - Category: {faq['category']}")
    
    faq_text = "\n".join(faq_list)
    
    prompt = f"""
    You are a FAQ matcher for MediCare Plus clinic.
    
    Here are all the available FAQs:
    {faq_text}
    
    User's question: "{message}"
    
    Your task:
    1. Find the FAQ that best answers the user's question
    2. If no FAQ is relevant, say "NONE"
    3. If a FAQ is relevant, respond with JUST the FAQ ID (e.g., FAQ001)
    
    Think step by step:
    - What is the user asking about?
    - Which FAQ(s) address this topic?
    - Choose the most specific and relevant FAQ
    
    Remember: Only respond with a FAQ ID or "NONE"
    """
    
    try:
        response = llm.invoke(prompt)
        faq_id = response.content.strip()
        
        if faq_id in CLINIC_FAQS:
            faq = CLINIC_FAQS[faq_id]
            return {
                "faq_id": faq_id,
                "question": faq["question"],
                "answer": faq["answer"],
                "category": faq["category"],
                "score": 10  # High confidence for LLM match
            }
        elif faq_id != "NONE":
            # LLM might have returned something else, try to parse
            for id in CLINIC_FAQS.keys():
                if id in response.content:
                    faq = CLINIC_FAQS[id]
                    return {
                        "faq_id": id,
                        "question": faq["question"],
                        "answer": faq["answer"],
                        "category": faq["category"],
                        "score": 8
                    }
    
    except Exception as e:
        print(f"LLM FAQ retrieval failed: {e}")
        # Fall back to keyword matching
        return retrieve_faq_answer_keyword(message)
    
    return None

def retrieve_faq_answer_keyword(message: str) -> Optional[Dict[str, Any]]:
    """
    Fallback keyword matching if LLM fails
    """
    message_lower = message.lower()
    best_match = None
    best_score = 0
    
    for faq_id, faq in CLINIC_FAQS.items():
        question = faq["question"].lower()
        answer = faq["answer"].lower()
        category = faq["category"].lower()
        
        score = 0
        
        # Check question words
        question_words = set(question.split())
        message_words = set(message_lower.split())
        common_words = question_words.intersection(message_words)
        score += len(common_words) * 2
        
        # Check if key terms are in the question
        if "specialt" in message_lower and "specialt" in question:
            score += 5
        if "appointment" in message_lower and "appointment" in question:
            score += 5
        if "payment" in message_lower and "payment" in question:
            score += 5
        if "insurance" in message_lower and "insurance" in question:
            score += 5
        if "hour" in message_lower and "hour" in question:
            score += 5
        if "location" in message_lower and "location" in question:
            score += 5
        
        if score > best_score:
            best_score = score
            best_match = {
                "faq_id": faq_id,
                "question": faq["question"],
                "answer": faq["answer"],
                "category": faq["category"],
                "score": score
            }
    
    if best_match and best_score >= 3:
        return best_match
    return None



print("✓ Layer 2: FAQ RAG logic implemented")

✓ Layer 2: FAQ RAG logic implemented


In [11]:
def retrieve_patient_data_with_llm(user_id: str, message: str) -> Optional[str]:
    """
    Enhanced patient data retrieval using LLM for natural language understanding
    """
    if user_id not in PATIENTS_DB:
        return None
    
    patient = PATIENTS_DB[user_id]
    first_name = patient["name"].split()[0]
    
    # Format patient data for the LLM
    patient_info = f"""
    Patient Information for {patient['name']}:
    
    Personal Details:
    - Email: {patient['email']}
    - Phone: {patient['phone']}
    - Date of Birth: {patient['date_of_birth']}
    - Membership Type: {patient['membership_type']}
    - Insurance Provider: {patient['insurance_provider']}
    - Primary Doctor: {patient['primary_doctor']}
    
    Medical Information:
    - Allergies: {', '.join(patient['allergies']) if patient['allergies'] else 'None'}
    - Last Visit: {patient['last_visit']}
    - Balance Due: €{patient['balance_due']:.2f}
    
    Upcoming Appointments:"""
    
    if patient['upcoming_appointments']:
        for i, appt in enumerate(patient['upcoming_appointments'], 1):
            patient_info += f"\n{i}. Date: {appt['date']}, Time: {appt['time']}, "
            patient_info += f"Doctor: {appt['doctor']}, Specialty: {appt['specialty']}, Status: {appt['status']}"
    else:
        patient_info += "\nNo upcoming appointments."
    
    prompt = f"""
    You are a medical assistant at MediCare Plus clinic. 
    You have access to this patient's information:
    
    {patient_info}
    
    Patient query: "{message}"
    
    Your task:
    1. Answer the patient's question based ONLY on the information above
    2. If the information is not available, say so
    3. Keep the answer concise and friendly
    4. Address the patient by their first name: {first_name}
    5. Include the specific data field(s) you're referencing
    
    Important: Do NOT make up any information. Only use what's provided.
    
    Format your response like this:
    [Answer: Your answer here] [Source: Patient Data - field_name]
    
    For example:
    [Answer: Hi Maria, your next appointment is on 2024-02-15 at 10:30 with Dr. Fernandez.] [Source: Patient Data - upcoming_appointments]
    
    Now respond to: "{message}"
    """
    
    try:
        response = llm.invoke(prompt)
        answer = response.content.strip()
        
        # Extract answer and source
        if "[Answer:" in answer and "[Source:" in answer:
            return answer
        else:
            # Fallback: format it properly
            return f"[Answer: {answer}] [Source: Patient Data - Comprehensive]"
            
    except Exception as e:
        print(f"LLM patient data retrieval failed: {e}")
        # Fall back to rule-based
        return retrieve_patient_data_rule_based(user_id, message)

def retrieve_patient_data_rule_based(user_id: str, message: str) -> Optional[str]:
    """
    Original rule-based patient data retrieval as fallback
    """
    if user_id not in PATIENTS_DB:
        return None

    patient = PATIENTS_DB[user_id]
    first_name = patient["name"].split()[0]
    message_lower = message.lower()

    # Balance queries
    if any(word in message_lower for word in ["balance", "owe", "bill", "due", "debt"]):
        balance = patient["balance_due"]
        if balance > 0:
            return f"[Answer: Hi {first_name}, you have a balance of €{balance:.2f}.] [Source: Patient Data - balance_due]"
        return f"[Answer: Hi {first_name}, you don't have any outstanding balance.] [Source: Patient Data - balance_due]"

    # Appointment queries
    if any(word in message_lower for word in ["appointment", "visit", "next", "upcoming", "schedule"]):
        appointments = patient["upcoming_appointments"]
        if appointments:
            next_appt = appointments[0]
            return (f"[Answer: Hi {first_name}, your next appointment is on {next_appt['date']} at {next_appt['time']} "
                   f"with {next_appt['doctor']} ({next_appt['specialty']}).] [Source: Patient Data - upcoming_appointments]")
        return f"[Answer: Hi {first_name}, you don't have any upcoming appointments.] [Source: Patient Data - upcoming_appointments]"

    # Doctor queries
    if any(word in message_lower for word in ["doctor", "primary", "physician", "dr"]):
        return f"[Answer: Hi {first_name}, your primary doctor is {patient['primary_doctor']}.] [Source: Patient Data - primary_doctor]"

    # Insurance queries
    if "insurance" in message_lower:
        return f"[Answer: Hi {first_name}, your insurance provider is {patient['insurance_provider']}.] [Source: Patient Data - insurance_provider]"

    # Allergy queries
    if any(word in message_lower for word in ["allerg", "react"]):
        allergies = patient["allergies"]
        if allergies:
            return f"[Answer: Hi {first_name}, your allergies are: {', '.join(allergies)}.] [Source: Patient Data - allergies]"
        return f"[Answer: Hi {first_name}, you have no recorded allergies.] [Source: Patient Data - allergies]"

    return None

print("✓ Layer 2: Patient DB RAG logic implemented")

✓ Layer 2: Patient DB RAG logic implemented


## Contextual Memory (Layer 3)

In [12]:
def resolve_references(message: str, conversation_history: List[Dict[str, str]]) -> Tuple[str, Optional[str]]:
    """
    LAYER 3: Contextual Memory
    Resolve pronouns and references ("his", "her", "that", "it", "the same doctor")
    by scanning conversation_history
    """
    if not conversation_history:
        return message, None

    # Extract entities from history
    mentioned_doctors = []
    mentioned_specialties = []

    doctor_pattern = r"Dr\. \w+"
    specialties = [
        "General Medicine", "Pediatrics", "Gynecology", "Dermatology",
        "Cardiology", "Traumatology", "Psychology", "Nutrition"
    ]

    for turn in conversation_history:
        content = turn.get("content", "")

        # Find doctors
        doctors = re.findall(doctor_pattern, content)
        mentioned_doctors.extend(doctors)

        # Find specialties
        for specialty in specialties:
            if specialty.lower() in content.lower():
                mentioned_specialties.append(specialty)

    # Get last mentioned doctor
    last_doctor = mentioned_doctors[-1] if mentioned_doctors else None

    # Resolve references in message
    resolved = message

    if last_doctor:
        # Replace pronouns with last mentioned doctor
        patterns = [
            (r'\bhis\b(?=\s+(specialty|doctor|appointment|availability))', f"{last_doctor}'s"),
            (r'\bher\b(?=\s+(specialty|doctor|appointment|availability))', f"{last_doctor}'s"),
            (r'\btheir\b(?=\s+(specialty|doctor|appointment|availability))', f"{last_doctor}'s"),
            (r'\bthat doctor\b', last_doctor),
            (r'\bthe same doctor\b', last_doctor),
            (r'\bit\b(?=\s+(specialty|available))', f"{last_doctor}'s specialty")
        ]

        for pattern, replacement in patterns:
            resolved = re.sub(pattern, replacement, resolved, flags=re.IGNORECASE)

    # Resolve specialty references
    if mentioned_specialties:
        last_specialty = mentioned_specialties[-1]
        resolved = re.sub(r'\bthat specialty\b', last_specialty, resolved, flags=re.IGNORECASE)

    return resolved, last_doctor

print("✓ Layer 3: Contextual memory implemented")

✓ Layer 3: Contextual memory implemented


## Escalation Checks (Layer 4)

In [13]:
def smart_check_escalation_with_llm(message: str, intent: Intent, rag_result: Optional[Any]) -> Tuple[bool, str]:
    """
    Use LLM to determine if we should escalate
    """
    message_lower = message.lower()
    
    # Always escalate emergencies
    if intent == Intent.EMERGENCY:
        return True, "emergency"
    
    # Use LLM for complex escalation decisions
    if intent == Intent.UNCLEAR or (intent == Intent.FAQ and not rag_result):
        prompt = f"""
        You are a medical chatbot assistant. Determine if this user query should be escalated to a human agent.
        
        User query: "{message}"
        
        Reasons to escalate:
        1. Asking about complex medical procedures we don't offer (transplants, major surgery, etc.)
        2. Complex medical advice or diagnosis
        3. Questions about specific medications or treatments we don't cover
        4. Complaints about staff or service quality
        5. Billing disputes or complex financial questions
        6. Legal or insurance disputes
        
        Reasons NOT to escalate:
        1. General questions about clinic operations (hours, location, services)
        2. Simple appointment or billing questions
        3. Questions we have FAQs for
        4. Simple information requests
        
        Should this be escalated to a human agent? Answer YES or NO.
        """
        
        try:
            response = llm.invoke(prompt)
            decision = response.content.strip().upper()
            
            if "YES" in decision:
                # Determine reason
                if any(word in message_lower for word in ["transplant", "surgery", "chemotherapy", "radiation"]):
                    return True, "complex_medical"
                elif any(word in message_lower for word in ["angry", "frustrated", "complaint", "sue"]):
                    return True, "negative_sentiment"
                else:
                    return True, "unanswerable"
        except Exception as e:
            print(f"LLM escalation check failed: {e}")
    
    # Rule-based fallback
    negative_keywords = [
        "angry", "frustrated", "furious", "unacceptable", "ridiculous",
        "terrible", "worst", "hate", "useless", "incompetent"
    ]
    
    if any(keyword in message_lower for keyword in negative_keywords):
        return True, "negative_sentiment"
    
    # Check for procedure questions
    procedure_keywords = ["transplant", "surgery", "chemotherapy", "dialysis", "bypass"]
    if any(keyword in message_lower for keyword in procedure_keywords):
        return True, "complex_medical"
    
    # Urgency
    urgency_keywords = ["asap", "urgent", "immediate", "right now", "today"]
    if any(keyword in message_lower for keyword in urgency_keywords):
        return True, "urgency"
    
    return False, "none"

## Output Formatting (Layer 5)

In [14]:
def format_response(content: str, source_type: str, source_id: str) -> str:
    """
    LAYER 5: Output Formatting
    Append source citations like [Source: FAQ001] or [Source: Patient Data - balance_due]
    """
    if source_type == "faq":
        return f"{content} [Source: {source_id}]"
    elif source_type == "patient_data":
        return f"{content} [Source: {source_id}]"
    else:
        return content

def format_escalation_message(reason: str) -> str:
    """Format appropriate escalation message based on reason"""
    messages = {
        "emergency": "  I understand this is an emergency. I'm immediately transferring you to a human agent. Please stay on the line. Transferring to agent...",
        "negative_sentiment": "I'm sorry to hear you're frustrated. Let me connect you with a human agent who can better assist you. Transferring to agent...",
        "unanswerable": "I'm sorry, I cannot find information related to your query in our database. Would you like to speak to a human agent who can help you further?",
        "urgency": "I understand this is urgent. Let me transfer you to a human agent who can provide immediate assistance. Transferring to agent..."
    }
    return messages.get(reason, "Transferring to agent...")

print("✓ Layer 5: Output formatting implemented")

✓ Layer 5: Output formatting implemented


## Main function with LangSmith tracing

In [15]:
def generate_response_llm_enhanced(
    user_id: str,
    current_message: str,
    conversation_history: List[Dict[str, str]]
) -> Dict[str, Any]:
    
    with tracing_v2_enabled(project_name="medicare-chatbot-full"):
        # LAYER 3: Contextual Memory
        resolved_message, last_doctor = resolve_references(current_message, conversation_history)
        
        # LAYER 1: Intent Detection
        intent = enhanced_detect_intent_with_llm(resolved_message, conversation_history)
        
        # LAYER 2: RAG Logic
        rag_result = None
        source_type = None
        source_id = None
        
        if intent == Intent.EMERGENCY:
            pass
            
        elif intent == Intent.PERSONAL_DATA:
            # Use LLM-enhanced patient data retrieval
            patient_response = retrieve_patient_data_with_llm(user_id, resolved_message)
            if patient_response:
                rag_result = patient_response
                source_type = "patient_data"
                # Extract source from response
                if "[Source:" in patient_response:
                    source_id = patient_response.split("[Source: ")[1].split("]")[0]
        
        elif intent == Intent.FAQ:
            # Use LLM for FAQ matching
            faq_match = retrieve_faq_with_llm(resolved_message)
            if faq_match:
                rag_result = faq_match["answer"]
                source_type = "faq"
                source_id = faq_match["faq_id"]
            else:
                # Fallback to keyword matching
                faq_match = retrieve_faq_answer_keyword(resolved_message)
                if faq_match:
                    rag_result = faq_match["answer"]
                    source_type = "faq"
                    source_id = faq_match["faq_id"]
        
        # LAYER 4: Escalation Checks
        should_escalate, escalation_reason = smart_check_escalation_with_llm(
            current_message, intent, rag_result
        )
        
        # LAYER 5: Output Formatting
        if should_escalate:
            response = format_escalation_message(escalation_reason)
        else:
            if rag_result:
                # Remove the formatting markers if present
                if rag_result.startswith("[Answer:") and rag_result.endswith("]"):
                    # Extract just the answer part
                    answer_part = rag_result.split("[Answer: ")[1].split("] [Source:")[0]
                    response = answer_part
                else:
                    response = rag_result
            else:
                response = "I'm sorry, I cannot find information related to your query. Would you like to speak to a human agent?"
                should_escalate = True
        
        return {
            "response": response,
            "escalate_to_human": should_escalate
        }


## Test all scenarios with LangSmith tracing

In [16]:
# Test the enhanced patient data retrieval
print("\n" + "="*70)
print("TESTING ENHANCED PATIENT DATA RETRIEVAL WITH LLM")
print("="*70)

test_queries = [
    ("What's my next appointment?", "Should show appointment details"),
    ("Who is my primary doctor?", "Should show primary doctor"),
    ("Do I owe any money?", "Should show balance"),
    ("What are my allergies?", "Should show allergies"),
    ("What's my insurance?", "Should show insurance provider"),
    ("When was my last visit?", "Should show last visit date"),
    ("Is Dr. Fernández available this week?", "Complex query - should use LLM"),
    ("Can I see a dermatologist soon?", "Complex query - should use LLM"),
]

for query, expected in test_queries:
    print(f"\nQuery: {query}")
    result = generate_response_llm_enhanced("P001", query, [])
    
    if result['escalate_to_human']:
        print(f"Result: ESCALATING - {result['response'][:80]}...")
    else:
        print(f"Result: {result['response']}")
    
    print("-" * 50)


TESTING ENHANCED PATIENT DATA RETRIEVAL WITH LLM

Query: What's my next appointment?
Result: 
--------------------------------------------------

Query: Who is my primary doctor?
Result: 
--------------------------------------------------

Query: Do I owe any money?
Result: 
--------------------------------------------------

Query: What are my allergies?
Result: Hi María, you are allergic to Penicillin.
--------------------------------------------------

Query: What's my insurance?
Result: [Answer: Hi María, your insurance provider is Sanitas.] [Source: Patient
--------------------------------------------------

Query: When was my last visit?
Result: 
--------------------------------------------------

Query: Is Dr. Fernández available this week?
Result: ESCALATING - I'm sorry, I cannot find information related to your query. Would you like to sp...
--------------------------------------------------

Query: Can I see a dermatologist soon?
Result: MediCare Plus offers: General Medicin

In [17]:
def test_llm_enhanced():
    print("\n" + "="*70)
    print("TESTING LLM-ENHANCED VERSION")
    print("="*70)
    
    test_cases = [
        ("What medical specialties are available?", "FAQ007 - Should match"),
        ("Do you do liver transplants?", "Should escalate"),
        ("How can I schedule an appointment?", "FAQ001 - Should match"),
        ("What is the cancellation policy?", "FAQ002 - Should match"),
        ("Where are you located?", "FAQ010 - Should match"),
        ("Can I get chemotherapy at your clinic?", "Should escalate"),
        ("I need to see a dermatologist", "FAQ007 - Should match specialties"),
        ("What payment methods do you accept?", "FAQ004 - Should match"),
        ("When was my last visit?", "Personal data - Should answer"),
        ("I'm having chest pain", "Should escalate as emergency"),
    ]
    
    for query, expected in test_cases:
        print(f"\nQuery: {query}")
        print(f"Expected: {expected}")
        
        # Determine user ID based on query
        user_id = "P001" if "my" in query.lower() or "last visit" in query.lower() else "P001"
        
        result = generate_response_llm_enhanced(user_id, query, [])
        
        if result['escalate_to_human']:
            print(f"Result: ESCALATING - {result['response'][:80]}...")
        else:
            print(f"Result: {result['response'][:100]}...")
        
        print("-" * 50)

# Run the test
test_llm_enhanced()

# Also test the multi-turn scenario
print("\n" + "="*70)
print("TESTING MULTI-TURN CONVERSATION")
print("="*70)

history = []
queries = [
    "What medical specialties do you have?",
    "Is Dr. López available for dermatology?",
    "When is my next appointment?"
]

for i, query in enumerate(queries):
    print(f"\nTurn {i+1}:")
    print(f"User: {query}")
    
    user_id = "P001" if "my" in query.lower() else "P001"
    result = generate_response_llm_enhanced(user_id, query, history)
    
    print(f"Bot: {result['response'][:120]}...")
    print(f"Escalate: {result['escalate_to_human']}")
    
    history.append({"role": "user", "content": query})
    history.append({"role": "bot", "content": result['response']})


TESTING LLM-ENHANCED VERSION

Query: What medical specialties are available?
Expected: FAQ007 - Should match
Result: MediCare Plus offers: General Medicine, Pediatrics, Gynecology, Dermatology, Cardiology, Traumatolog...
--------------------------------------------------

Query: Do you do liver transplants?
Expected: Should escalate
Result: ESCALATING - Transferring to agent......
--------------------------------------------------

Query: How can I schedule an appointment?
Expected: FAQ001 - Should match
Result: You can schedule an appointment through our online portal at medicare-plus.com/appointments, by call...
--------------------------------------------------

Query: What is the cancellation policy?
Expected: FAQ002 - Should match
Result: Appointments must be cancelled at least 24 hours in advance to avoid a €20 cancellation fee. Premium...
--------------------------------------------------

Query: Where are you located?
Expected: FAQ010 - Should match
Result: We are located at C

In [18]:
def run_all_tests():
    """Run all required test scenarios"""

    print("=" * 80)
    print("AI CUSTOMER SERVICE BOT - 5-LAYER ARCHITECTURE TESTS")
    print("=" * 80)

    # Test 1: The Happy Path
    print("\n Test 1: The Happy Path")
    print("-" * 40)
    result =   generate_response_llm_enhanced("P001", "How do I pay?", [])
    print(f"User (P001): How do I pay?")
    print(f"Bot: {result['response']}")
    print(f"Escalate: {result['escalate_to_human']}")

    # Test 2: The Personal Query
    print("\n Test 2: The Personal Query")
    print("-" * 40)
    result = generate_response_llm_enhanced("P003", "Do I owe anything?", [])
    print(f"User (P003): Do I owe anything?")
    print(f"Bot: {result['response']}")
    print(f"Escalate: {result['escalate_to_human']}")

    # Test 3: The Emergency
    print("\n Test 3: The Emergency")
    print("-" * 40)
    result = generate_response_llm_enhanced("P002", "I'm having severe chest pain.", [])
    print(f"User (P002): I'm having severe chest pain.")
    print(f"Bot: {result['response']}")
    print(f"Escalate: {result['escalate_to_human']}")

    # Test 4: The Hallucination Trap
    print("\n Test 4: The Hallucination Trap")
    print("-" * 40)
    result = generate_response_llm_enhanced("P001", "Do you do liver transplants?", [])
    print(f"User (P001): Do you do liver transplants?")
    print(f"Bot: {result['response']}")
    print(f"Escalate: {result['escalate_to_human']}")

    # Test 5: Memory (Multi-turn)
    print("\n Test 5: Memory (Multi-turn)")
    print("-" * 40)

    # Start conversation
    history = []

    # Turn 1
    result1 = generate_response_llm_enhanced("P001", "Who is my primary doctor?", history)
    print(f"User (T1): Who is my primary doctor?")
    print(f"Bot: {result1['response']}")

    # Update history
    history.append({"role": "user", "content": "Who is my primary doctor?"})
    history.append({"role": "bot", "content": result1["response"]})

    # Turn 2
    result2 = generate_response_llm_enhanced("P001", "What is his specialty?", history)
    print(f"\nUser (T2): What is his specialty?")
    print(f"Bot: {result2['response']}")
    print(f"Escalate: {result2['escalate_to_human']}")

    print("\n" + "=" * 80)
    print("ALL TESTS COMPLETED - Check LangSmith dashboard for traces!")
    print("=" * 80)

# Run tests
run_all_tests()

AI CUSTOMER SERVICE BOT - 5-LAYER ARCHITECTURE TESTS

 Test 1: The Happy Path
----------------------------------------
User (P001): How do I pay?
Bot: We accept cash, credit/debit cards (Visa, Mastercard, American Express), bank transfers, and payment through major insurance providers (Sanitas, Adeslas, Mapfre, DKV, Asisa). Payment plans are available for treatments exceeding €500.
Escalate: False

 Test 2: The Personal Query
----------------------------------------
User (P003): Do I owe anything?
Bot: 
Escalate: False

 Test 3: The Emergency
----------------------------------------
User (P002): I'm having severe chest pain.
Bot:   I understand this is an emergency. I'm immediately transferring you to a human agent. Please stay on the line. Transferring to agent...
Escalate: True

 Test 4: The Hallucination Trap
----------------------------------------
User (P001): Do you do liver transplants?
Bot: Transferring to agent...
Escalate: True

 Test 5: Memory (Multi-turn)
------------------

## Additional test cases for comprehensive validation

In [19]:
def run_additional_tests():
    """Run additional edge cases and scenarios"""

    print("\n" + "=" * 80)
    print("ADDITIONAL TEST SCENARIOS")
    print("=" * 80)

    # Test: Negative sentiment
    print("\nTest: Negative Sentiment")
    print("-" * 40)
    result = generate_response_llm_enhanced("P001", "I'm really angry about your service!", [])
    print(f"User: I'm really angry about your service!")
    print(f"Bot: {result['response']}")
    print(f"Escalate: {result['escalate_to_human']}")

    # Test: Insurance query
    print("\nTest: Insurance Query")
    print("-" * 40)
    result = generate_response_llm_enhanced("P002", "What insurance do I have?", [])
    print(f"User (P002): What insurance do I have?")
    print(f"Bot: {result['response']}")
    print(f"Escalate: {result['escalate_to_human']}")

    # Test: Appointment scheduling FAQ
    print("\n Test: Appointment Scheduling FAQ")
    print("-" * 40)
    result = generate_response_llm_enhanced("P001", "How can I schedule an appointment?", [])
    print(f"User: How can I schedule an appointment?")
    print(f"Bot: {result['response']}")
    print(f"Escalate: {result['escalate_to_human']}")

    # Test: Unknown user ID
    print("\n Test: Unknown User ID")
    print("-" * 40)
    result = generate_response_llm_enhanced("P999", "What is my balance?", [])
    print(f"User (P999): What is my balance?")
    print(f"Bot: {result['response']}")
    print(f"Escalate: {result['escalate_to_human']}")

    # Test: Complex multi-turn
    print("\n Test: Complex Multi-turn Conversation")
    print("-" * 40)
    history = []

    # Conversation flow
    questions = [
        "What medical specialties are available?",
        "Is Dr. López available for that?",
        "When is my next appointment?"
    ]

    for i, question in enumerate(questions):
        result = generate_response_llm_enhanced("P001", question, history)
        print(f"\nTurn {i+1}:")
        print(f"User: {question}")
        print(f"Bot: {result['response'][:100]}..." if len(result['response']) > 100 else f"Bot: {result['response']}")

        history.append({"role": "user", "content": question})
        history.append({"role": "bot", "content": result["response"]})

    print("\n" + "=" * 80)
    print("ADDITIONAL TESTS COMPLETED")
    print("=" * 80)

run_additional_tests()


ADDITIONAL TEST SCENARIOS

Test: Negative Sentiment
----------------------------------------
User: I'm really angry about your service!
Bot: I'm sorry to hear you're frustrated. Let me connect you with a human agent who can better assist you. Transferring to agent...
Escalate: True

Test: Insurance Query
----------------------------------------
User (P002): What insurance do I have?
Bot: 
Escalate: False

 Test: Appointment Scheduling FAQ
----------------------------------------
User: How can I schedule an appointment?
Bot: You can schedule an appointment through our online portal at medicare-plus.com/appointments, by calling our reception at +34 911 234 567 (Monday to Friday, 8:00-20:00), or through this chat by requesting to speak with our scheduling team.
Escalate: False

 Test: Unknown User ID
----------------------------------------
User (P999): What is my balance?
Bot: I'm sorry, I cannot find information related to your query. Would you like to speak to a human agent?
Escalate: